# Week 2: Provider-Agnostic LLM Client (Gemini + Ollama/Llama 3.2)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (CPU also works, just slower)
2. Add a Colab **Secret** (key icon, left sidebar): name `GEMINI_API_KEY`, paste your key from aistudio.google.com/apikey, and turn on **Notebook access**

Run all cells in order. If the runtime restarts, re-run everything from the top.

## Step 1: Project folder

In [ ]:
import os
os.makedirs("/content/week-2", exist_ok=True)
%cd /content/week-2
!pwd

## Step 2: Install Ollama

In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

## Step 3: Start the Ollama server in the background

In [ ]:
import subprocess, time, requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.ok:
            print("Server says:", r.text)
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")

## Step 4: Download Llama 3.2 (~2 GB)

In [ ]:
!ollama pull llama3.2
!ollama list

## Step 5: Raw test of the local model

In [ ]:
!curl -s http://localhost:11434/api/generate -d '{"model": "llama3.2", "prompt": "Say hello in one short sentence.", "stream": false}'

## Step 6: Create `.env` from the Colab secret

In [ ]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

with open("/content/week-2/.env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')

print(".env created. Key length:", len(key))

## Step 7: requirements.txt and install

In [ ]:
%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests

In [ ]:
!pip install -q -r requirements.txt
from google import genai
print("google-genai imported OK")

## Step 8: `llm_client.py`

In [ ]:
%%writefile llm_client.py
from abc import ABC, abstractmethod
from google import genai
import requests


# Common interface: every provider must implement generate()
class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass


# Gemini cloud provider
class GeminiProvider(LLMProvider):

    FALLBACK_MODELS = ["gemini-3.5-flash", "gemini-2.5-flash"]

    def __init__(self, api_key, model="gemini-3.6-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        models_to_try = [self.model] + [
            m for m in self.FALLBACK_MODELS if m != self.model
        ]
        last_error = None
        for model in models_to_try:
            try:
                response = self.client.models.generate_content(
                    model=model,
                    contents=prompt
                )
                self.model = model  # remember the one that worked
                return response.text
            except Exception as e:
                last_error = e
        raise RuntimeError(f"All Gemini models failed. Last error: {last_error}")


# Ollama local provider
class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=300
        )
        response.raise_for_status()
        return response.json()["response"]


# Common client: the rest of the program only talks to this class
class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)

## Step 9: `main.py`

In [ ]:
%%writefile main.py
import os
from dotenv import load_dotenv
from llm_client import (
    GeminiProvider,
    OllamaProvider,
    LLMClient
)

load_dotenv(override=True)

provider_name = input(
    "Choose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Gemini API key not found in .env")
    provider = GeminiProvider(api_key)

elif provider_name == "ollama":
    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)

prompt = input("Enter your question: ")
answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)

## Step 10: Unit test with a fake provider

In [ ]:
%%writefile test_client.py
import unittest
from llm_client import LLMClient


# Fake provider for testing
class FakeProvider:
    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."


class TestLLMClient(unittest.TestCase):

    def test_generate(self):
        provider = FakeProvider()
        client = LLMClient(provider)
        result = client.generate("Explain AI")
        self.assertEqual(
            result,
            "AI stands for Artificial Intelligence."
        )


if __name__ == "__main__":
    unittest.main()

In [ ]:
!python -m pytest test_client.py -v

## Step 11: Run the program

Use `%run` (not `!python`) so `input()` prompts work. Run once and type `ollama`, then run again and type `gemini`.

In [ ]:
%run main.py